# Page View 로그 SQL 생성
- pageName: 홈, 로그인, 회원가입, 상품목록, 장바구니, 주문결제, 주문완료, 마이페이지, 회원탈퇴
- user_id: 1~100 (로그인) / null (비로그인)
- user_login_id: user0001~user0100 (로그인) / null (비로그인)
- client_uuid: 세션마다 고유 UUID

In [1]:
import random
import json
import uuid
from datetime import datetime, timedelta

In [2]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
PAGE_NAMES        = ['홈', '로그인', '회원가입', '상품목록', '장바구니', '주문결제', '주문완료', '마이페이지']
WITHDRAWAL_PAGE   = '회원탈퇴'
WITHDRAWAL_RATIO  = 0.01  # 회원탈퇴 페이지 비율

START_DATE = datetime(2025, 6, 1, 0, 0, 0)
END_DATE   = datetime(2026, 6, 16, 23, 59, 59)
ROW_COUNT  = 500   # 생성할 로그 수

# 비로그인 사용자 비율 (0.0 ~ 1.0)
ANONYMOUS_RATIO = 0.3

In [3]:
def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

def format_kst(dt):
    """event_timestamp 포맷 (KST +09:00)"""
    return dt.strftime('%Y-%m-%dT%H:%M:%S.') + f"{dt.microsecond // 1000:03d}+09:00"

def format_history_ts(dt):
    """history_timestamp 포맷 (마이크로초 포함)"""
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')

In [4]:
rows = []

for _ in range(ROW_COUNT):
    # event_timestamp 기준으로 먼저 뽑고, history_timestamp = event_ts + 1초 (서버 수신이 더 늦음)
    event_ts   = random_datetime(START_DATE, END_DATE)
    history_ts = event_ts + timedelta(seconds=1)

    # 회원탈퇴 페이지는 1%, 나머지 8개 페이지는 99%를 균등 분배
    if random.random() < WITHDRAWAL_RATIO:
        page_name = WITHDRAWAL_PAGE
    else:
        page_name = random.choice(PAGE_NAMES)
    dwell_time = random.randint(1, 30)

    # 세션마다 고유 UUID (비로그인 사용자도 UUID는 항상 존재)
    client_uuid = str(uuid.uuid4())

    # 비로그인 사용자 여부
    is_anonymous = random.random() < ANONYMOUS_RATIO

    if is_anonymous:
        user_id       = None
        user_login_id = None
    else:
        user_id       = random.randint(1, 100)
        user_login_id = f'user{user_id:04d}'

    json_log = json.dumps({
        'event_name':      'page_view',
        'pageName':        page_name,
        'dwellTime':       dwell_time,
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(event_ts)
    }, ensure_ascii=False)

    rows.append((history_ts, json_log))

print(f'✅ {ROW_COUNT}개 page_view 로그 생성 완료')

✅ 500개 page_view 로그 생성 완료


In [5]:
# SQL 생성 및 저장
lines  = ["INSERT INTO first_save_history (history_timestamp, json_log) VALUES"]
values = []

for history_ts, json_log in rows:
    ts_str  = format_history_ts(history_ts)
    escaped = json_log.replace("'", "''")
    values.append(f"  ('{ts_str}', '{escaped}')")

lines.append(',\n'.join(values) + ';')
sql = '\n'.join(lines)

with open('page_view_logs.sql', 'w', encoding='utf-8') as f:
    f.write(sql)

print(f'✅ {ROW_COUNT}개 page_view 로그 SQL 생성 완료 → page_view_logs.sql')

✅ 500개 page_view 로그 SQL 생성 완료 → page_view_logs.sql


In [6]:
# ── 미리보기 ──
print('=== PAGE VIEW SQL (앞 500자) ===')
print(sql[:500])

=== PAGE VIEW SQL (앞 500자) ===
INSERT INTO first_save_history (history_timestamp, json_log) VALUES
  ('2025-10-01 02:00:46.000000', '{"event_name": "page_view", "pageName": "회원가입", "dwellTime": 21, "user_id": 76, "user_login_id": "user0076", "client_uuid": "32a49ee0-9289-49da-9b3f-35ab1287d3b4", "event_timestamp": "2025-10-01T02:00:45.000+09:00"}'),
  ('2025-06-11 09:47:29.000000', '{"event_name": "page_view", "pageName": "로그인", "dwellTime": 16, "user_id": null, "user_login_id": null, "client_uuid": "8fe99c1d-2ff3-4282-a567-8
